# Renon 3D Gaussian Splatting — Probelauf (Kaggle)

Trainiert 3DGS auf dem vorbereiteten COLMAP-Datensatz (20 posierte Pinhole-Bilder
+ 300k LiDAR-Initpunkte). Posen exakt aus den E57-Bloecken, gegen die Fotos geprueft.

## Vor dem Start — zwei Einstellungen rechts im Panel:
1. **Settings → Accelerator → GPU T4 x2** (oder P100). Ohne GPU laeuft nichts.
2. **Settings → Internet → On.** Wird fuer `git clone` und `pip install` gebraucht;
   Kaggle hat es standardmaessig AUS.

## Datensatz bereitstellen (statt Upload wie bei Colab):
Rechts **Add Input → Datasets → New Dataset** und `colmap_renon4.zip` (15 MB, aus
`data/Renon/`) hochladen. Kaggle haengt es unter `/kaggle/input/<name>/` ein;
Zelle 3 findet die Zip dort automatisch.

Dauer auf T4: ~20–40 Min fuer 30k Iterationen. Ergebnis `renon_gaussians.ply`
liegt danach in `/kaggle/working/` und ist im **Output**-Tab herunterladbar
(dann per Drag&Drop in https://playcanvas.com/supersplat/editor).

In [ ]:
# 1) Vorflugkontrolle: GPU da? Welche torch/CUDA-Kombination?
import torch, os
print('torch       :', torch.__version__)
print('CUDA (torch):', torch.version.cuda)
assert torch.cuda.is_available(), 'Keine GPU — Settings > Accelerator > GPU setzen!'
print('GPU         :', torch.cuda.get_device_name(0))
cap = torch.cuda.get_device_capability(0)
os.environ['TORCH_CUDA_ARCH_LIST'] = f'{cap[0]}.{cap[1]}'   # T4=7.5, P100=6.0
print('Baue fuer Compute Capability', os.environ['TORCH_CUDA_ARCH_LIST'])
import urllib.request
try:
    urllib.request.urlopen('https://github.com', timeout=10)
    print('Internet: OK')
except Exception:
    raise SystemExit('Kein Internet — Settings > Internet > On setzen!')

In [ ]:
# 2) 3DGS klonen + CUDA-Submodule bauen (~4-6 Min)
#    DREI Submodule -- fused-ssim ist seit Ende 2024 Pflicht, sonst bricht
#    train.py sofort mit ModuleNotFoundError ab.
%cd /kaggle/working
!rm -rf gaussian-splatting
!git clone --recursive --depth 1 https://github.com/graphdeco-inria/gaussian-splatting
%cd /kaggle/working/gaussian-splatting
!pip install -q plyfile opencv-python joblib
!pip install -q --no-build-isolation \
    ./submodules/diff-gaussian-rasterization \
    ./submodules/simple-knn \
    ./submodules/fused-ssim
import importlib
for m in ('diff_gaussian_rasterization', 'simple_knn._C', 'fused_ssim'):
    importlib.import_module(m); print('OK:', m)

In [ ]:
# 3) Datensatz aus dem angehaengten Kaggle-Dataset finden und entpacken
import glob, os, zipfile
zips = glob.glob('/kaggle/input/**/colmap_renon4.zip', recursive=True)
assert zips, 'colmap_renon4.zip nicht gefunden — rechts als Dataset hinzufuegen (Add Input).'
print('Zip:', zips[0])
os.makedirs('/kaggle/working/renon', exist_ok=True)
with zipfile.ZipFile(zips[0]) as z:
    z.extractall('/kaggle/working/renon')
root = '/kaggle/working/renon'
if not os.path.isdir(f'{root}/sparse'):
    inner = glob.glob(f'{root}/*/sparse')
    if inner: root = os.path.dirname(inner[0])
n = len(glob.glob(f'{root}/images/*.jpg'))
print('Datensatz-Wurzel:', root, '| Bilder:', n, '| sparse/0:', os.listdir(f'{root}/sparse/0'))
assert n == 20, f'20 Bilder erwartet, {n} gefunden'
DATA = root

In [ ]:
# 4) Training. -r 2 rechnet auf 1024 px, --data_device cpu haelt die Bilder im RAM.
%cd /kaggle/working/gaussian-splatting
!python train.py -s {DATA} -m /kaggle/working/output/renon \
    --iterations 30000 -r 2 --data_device cpu

In [ ]:
# 5) Ergebnis nach /kaggle/working/ (im Output-Tab herunterladbar)
import os, shutil
ply = '/kaggle/working/output/renon/point_cloud/iteration_30000/point_cloud.ply'
assert os.path.isfile(ply), 'Kein Ergebnis — Trainingslog oben pruefen'
print(f'{os.path.getsize(ply)/1e6:.1f} MB Gaussians')
shutil.copyfile(ply, '/kaggle/working/renon_gaussians.ply')
# Aufraeumen, damit der Output-Tab schlank bleibt (nur das PLY behalten)
shutil.rmtree('/kaggle/working/gaussian-splatting', ignore_errors=True)
shutil.rmtree('/kaggle/working/renon', ignore_errors=True)
print('-> /kaggle/working/renon_gaussians.ply  (Output-Tab -> Download)')

---
## Ausweichweg, falls Zelle 2 nicht baut

Bricht der CUDA-Build (neuere torch/CUDA-Kombination), ersetzt die naechste Zelle
Bau **und** Training ueber nerfstudio/gsplat mit vorkompilierten Wheels.

In [ ]:
# ALTERNATIVE zu Zelle 2+4 — nur bei Submodul-Build-Fehler
!pip install -q nerfstudio
!ns-train splatfacto --data {DATA} --max-num-iterations 30000 \
    --viewer.quit-on-train-completion True colmap --downscale-factor 2
import glob, shutil
cfg = sorted(glob.glob('/kaggle/working/outputs/**/config.yml', recursive=True))[-1]
!ns-export gaussian-splat --load-config {cfg} --output-dir /kaggle/working/export
shutil.copyfile('/kaggle/working/export/splat.ply', '/kaggle/working/renon_gaussians.ply')
print('-> /kaggle/working/renon_gaussians.ply')

## Ergebnis ansehen

`renon_gaussians.ply` aus dem **Output**-Tab herunterladen, dann per Drag&Drop in
**[SuperSplat](https://playcanvas.com/supersplat/editor)** — sofort frei begehbar,
fotorealistisch mit Parallaxe.

**Worauf zu achten ist:** die vier Standpunkte liegen fast kollinear auf ~10 m.
Quer dazu fehlt Parallaxe -> dort Schlieren; entlang des Wegs sollte es gut
aussehen. Daran laesst sich ablesen, wie viele Standpunkte ein voller Lauf braucht.